# Laboratorium 2: Obliczenia Numeryczne i Manipulacja Danymi
**Kierunek:** Sztuczna inteligencja w biznesie (Semestr 1)  
**Narzędzia:** `numpy`, `pandas`  
**Cel:** Opanowanie wektoryzacji danych, pracy z tabelami DataFrame, czyszczenia "brudnych" danych biznesowych oraz zaawansowanych agregacji.

In [ ]:
# [KOMÓRKA STARTOWA] Importujemy podstawowe biblioteki Data Science
import numpy as np
import pandas as pd

print(f"Wersja NumPy: {np.__version__}")
print(f"Wersja Pandas: {pd.__version__}")

### 1.1. Dlaczego NumPy, a nie zwykłe listy w Pythonie?
Zwykłe listy w Pythonie są elastyczne, ale **bardzo wolne** przy dużych zbiorach danych, ponieważ każdy element jest osobnym obiektem w pamięci.
Tablica NumPy (`ndarray`):
1. Przechowuje elementy **jednego typu** (najczęściej liczby).
2. Jest zlokalizowana w ciągłym bloku pamięci RAM.
3. Pozwala na **wektoryzację** – wykonanie operacji na milionie liczb jednocześnie bez użycia pętli `for`.

In [ ]:
# [PRZYKŁAD: Porównanie wydajności]
import time

# Zwykła lista w Pythonie (1 milion elementów)
lista_liczb = list(range(1_000_000))
start = time.time()
wynik_lista = [x * 1.23 for x in lista_liczb] # doliczamy podatek w pętli
czas_lista = time.time() - start

# Tablica NumPy (1 milion elementów)
tablica_np = np.arange(1_000_000)
start = time.time()
wynik_np = tablica_np * 1.23 # Wektoryzacja! Bez pętli for
czas_np = time.time() - start

print(f"Czas wykonania w czystym Pythonie: {czas_lista:.4f} s")
print(f"Czas wykonania w NumPy:           {czas_np:.4f} s")
print(f"-> NumPy jest około {czas_lista / czas_np:.1f}x szybszy!")

### 1.2. Tworzenie tablic i operacje statystyczne
W NumPy operacje wykonujemy na całych wektorach lub macierzach.

In [ ]:
# [PRZYKŁAD]
# Tworzenie tablicy z cenami sprzedaży
ceny = np.array([120.0, 450.0, 89.90, 1200.0, 310.0])

print("Ceny:", ceny)
print("Typ danych w tablicy:", ceny.dtype)
print("Średnia cena:", np.mean(ceny))
print("Odchylenie standardowe:", np.std(ceny))
print("Cena maksymalna:", np.max(ceny))

# Podwyżka wszystkich cen o 10% (Broadcasting)
ceny_nowe = ceny * 1.10
print("Ceny po podwyżce 10%:", np.round(ceny_nowe, 2))

### 1.3. Maski logiczne (Filtrowanie tablic)
Możemy w ułamku sekundy odfiltrować liczby spełniające określony warunek biznesowy.

In [ ]:
# [PRZYKŁAD]
# Szukamy tylko transakcji o wartości powyżej 200 PLN
maska = ceny > 200.0
print("Maska logiczna (True/False):", maska)
print("Tylko drogie produkty:", ceny[maska])

---
### ZADANIE 1 (Samodzielnie dla studentów - 10 min)
Dział analityczny banku otrzymał tablicę z kwotami wniosków kredytowych (w tys. PLN):
`wnioski = np.array([15, 120, 45, 8, 250, 60, 15, 300, 40, 95])`

Wykonaj bez używania pętli `for`:
1. Oblicz średnią (`np.mean`) oraz medianę (`np.median`) kwot wniosków.
2. Odfiltruj i wyświetl tylko wnioski **o wysokim ryzyku**, czyli kwoty powyżej **100 tys. PLN**.
3. Bank nakłada prowizję przygotowawczą w wysokości **2.5%**. Oblicz nową tablicę `prowizje`, zawierającą kwotę prowizji dla każdego wniosku.

In [ ]:
# [ROZWIĄZANIE ZADANIA 1 ]
wnioski = np.array([15, 120, 45, 8, 250, 60, 15, 300, 40, 95])

print(f"Średnia kwota wniosku: {np.mean(wnioski):.2f} tys. PLN")
print(f"Mediana kwot:         {np.median(wnioski):.2f} tys. PLN")

wysokie_ryzyko = wnioski[wnioski > 100]
print("Wnioski powyżej 100 tys. PLN:", wysokie_ryzyko)

prowizje = wnioski * 0.025
print("Prowizje bankowe (tys. PLN):", np.round(prowizje, 3))

### 2.1. DataFrame – serce analizy danych
- `pd.Series` to pojedyncza kolumna z indeksem.
- `pd.DataFrame` to cała 2-wymiarowa tabela (zbiór kolumn `Series`).

In [ ]:
# [PRZYKŁAD: Generujemy realistyczną tabelę transakcji]
dane_sklepu = {
    "ID_Zamowienia": [101, 102, 103, 104, 105, 106],
    "Klient": ["Firma_A", "Firma_B", "Firma_C", "Firma_A", "Firma_D", "Firma_B"],
    "Kraj": ["Polska", "Niemcy", "Polska", "Polska", "Francja", "Niemcy"],
    "Kategoria": ["Elektronika", "Biuro", "Elektronika", "Oprogramowanie", "Biuro", "Elektronika"],
    "Wartosc_PLN": [4500.0, 320.0, 1200.0, 8900.0, 450.0, 6100.0],
    "Sztuki": [2, 10, 1, 5, 15, 3]
}

df = pd.DataFrame(dane_sklepu)
df

### 2.2. Szybka inspekcja i wymiary tabeli
Przed przystąpieniem do analizy ZAWSZE wykonujemy 3 podstawowe komendy diagnostyczne.

In [ ]:
# [PRZYKŁAD]
print("1. Kształt tabeli (wiersze, kolumny):", df.shape)
print("\n2. Informacje techniczne i typy danych:")
df.info()

print("\n3. Podstawowe statystyki opisowe dla liczb:")
df.describe()

### 2.3. Selekcja kolumn i wierszy: `.loc[]` vs `.iloc[]`
- `df['Nazwa']` lub `df[['K1', 'K2']]` – wybór pojedynczej lub wielu kolumn.
- `df.iloc[wiersze, kolumny]` – wybieranie po **indeksach liczbowych** (od 0 do n).
- `df.loc[wiersze, kolumny]` – wybieranie po **etykietach / nazwach kolumn**.

In [ ]:
# [PRZYKŁAD]
# Pobranie pierwszych 3 wierszy i kolumn od 1 do 4 po indeksach liczbowych:
print("--- Wybór iloc (wiersze 0:3, kolumny 1:4) ---")
print(df.iloc[0:3, 1:4])

# Filtrowanie biznesowe z warunkiem: Zamówienia z Polski powyżej 2000 PLN
warunek = (df["Kraj"] == "Polska") & (df["Wartosc_PLN"] > 2000)
print("\n--- Zamówienia z Polski > 2000 PLN ---")
print(df[warunek])

---
### ZADANIE 2 (Samodzielnie dla studentów - 10 min)
Korzystając z tabeli `df` (wygenerowanej powyżej):
1. Wyświetl tylko dwie kolumny: `Klient` oraz `Wartosc_PLN`.
2. Przefiltruj tabelę, aby znaleźć wszystkie transakcje, które dotyczą kategorii **"Elektronika"** LUB zostały zrealizowane dla klienta **"Firma_B"** (użyj operatora `|`).
3. Oblicz średnią wartość zamówienia dla kategorii "Elektronika".

In [ ]:
# [ROZWIĄZANIE ZADANIA 2]
# 1. Dwie kolumny
print(df[["Klient", "Wartosc_PLN"]])

# 2. Filtr Elektronika LUB Firma_B
filtr = (df["Kategoria"] == "Elektronika") | (df["Klient"] == "Firma_B")
wynik_filtr = df[filtr]
print("\nPrzefiltrowane zamówienia:")
print(wynik_filtr)

# 3. Średnia dla Elektroniki
srednia_elektronika = df[df["Kategoria"] == "Elektronika"]["Wartosc_PLN"].mean()
print(f"\nŚrednia wartość zamówienia Elektroniki: {srednia_elektronika:.2f} PLN")

### 3.1. Praca z brakami danych (`NaN`) i duplikatami
W realnym świecie dane przychodzą z brakami, duplikatami i błędnymi typami.
- `.isna().sum()` – zlicza braki w każdej kolumnie.
- `.dropna()` – usuwa wiersze z brakami.
- `.fillna(wartość)` – uzupełnia braki (np. średnią lub medianą).
- `.drop_duplicates()` – usuwa powtórzone wiersze.

In [ ]:
# [PRZYKŁAD: Tworzymy "brudną" bazę klientów]
brudne_dane = {
    "ID": [1, 2, 3, 4, 4, 5, 6],
    "Wiek": [25, np.nan, 42, 30, 30, np.nan, 55], # braki NaN
    "Miasto": ["Warszawa", "Kraków", np.nan, "Wrocław", "Wrocław", "Gdańsk", "Warszawa"],
    "Roczny_Dochod_PLN": [60000, 85000, 110000, 72000, 72000, 50000, np.nan]
}

df_brudny = pd.DataFrame(brudne_dane)
print("Tabela początkowa (z brakami i duplikatem):")
print(df_brudny)

print("\nLiczba braków w kolumnach:")
print(df_brudny.isna().sum())

In [ ]:
# [PRZYKŁAD: Proces czyszczenia]
# 1. Usunięcie duplikatów
df_czysty = df_brudny.drop_duplicates().copy()

# 2. Imputacja braków w wieku (uzupełniamy medianą)
mediana_wieku = df_czysty["Wiek"].median()
df_czysty["Wiek"] = df_czysty["Wiek"].fillna(mediana_wieku)

# 3. Uzupełnienie braków w Mieście wartością "Nieznane"
df_czysty["Miasto"] = df_czysty["Miasto"].fillna("Nieznane")

# 4. Usunięcie wierszy, gdzie brakuje dochodu
df_czysty = df_czysty.dropna(subset=["Roczny_Dochod_PLN"])

print("\nTabela po wyczyszczeniu:")
print(df_czysty)

### 3.2. Tworzenie nowych cech (Feature Engineering)
W analityce i AI często tworzymy nowe kolumny na podstawie istniejących.

In [ ]:
# [PRZYKŁAD]
# Dodajemy miesięczny dochód oraz flagę wieku powyżej 35 lat
df_czysty["Miesieczny_Dochod"] = np.round(df_czysty["Roczny_Dochod_PLN"] / 12, 2)
df_czysty["Czy_Senior_35+"] = df_czysty["Wiek"] >= 35

df_czysty

---
### ZADANIE 3 (Samodzielnie dla studentów - 10 min)
Firma logistyczna przesłała dane o przesyłkach z błędami:

In [ ]:
dane_logistyka = {
    "Paczka_ID": ["P1", "P2", "P3", "P4", "P4", "P5"],
    "Waga_kg": [12.5, np.nan, 5.0, 20.0, 20.0, 1.2],
    "Odleglosc_km": [150, 300, 50, 450, 450, np.nan],
    "Cena_Bazowa": [50.0, 80.0, 30.0, 120.0, 120.0, 20.0]
}
df_log = pd.DataFrame(dane_logistyka)
print(df_log)

Wykonaj następujące kroki czyszczenia i transformacji:
1. Usuń duplikaty z tabeli `df_log`.
2. Uzupełnij brak w kolumnie `Waga_kg` średnią wagą z pozostałych paczek.
3. Usuń wiersze, w których brakuje `Odleglosc_km`.
4. Utwórz nową kolumnę `Koszt_Calkowity_PLN`, która wynosi: `Cena_Bazowa + (Waga_kg * 1.5)`.

In [ ]:
# [ROZWIĄZANIE ZADANIA 3]
# 1. Duplikaty
df_log_clean = df_log.drop_duplicates().copy()

# 2. Uzupełnienie wagi średnią
srednia_waga = df_log_clean["Waga_kg"].mean()
df_log_clean["Waga_kg"] = df_log_clean["Waga_kg"].fillna(srednia_waga)

# 3. Usunięcie braków odległości
df_log_clean = df_log_clean.dropna(subset=["Odleglosc_km"])

# 4. Nowa kolumna kosztu
df_log_clean["Koszt_Calkowity_PLN"] = df_log_clean["Cena_Bazowa"] + (df_log_clean["Waga_kg"] * 1.5)

print("Wyczyszczona tabela logistyczna:")
print(df_log_clean)

### 4.1. Grupowanie danych – odpowiednik tabeli przestawnej i SQL GROUP BY
Metoda `.groupby()` pozwala podzielić dane na grupy, zastosować funkcję agregującą (np. `sum`, `mean`, `count`) i połączyć wyniki.

In [ ]:
# [PRZYKŁAD: Powrót do tabeli sprzedaży df]
# Obliczamy łączną i średnią sprzedaż wg Krajów
raport_kraj = df.groupby("Kraj")["Wartosc_PLN"].agg(
    Liczba_Zamowien='count',
    Suma_Sprzedazy='sum',
    Srednia_Wartosc='mean'
).reset_index()

# Sortowanie malejąco wg sumy sprzedaży
raport_kraj = raport_kraj.sort_values(by="Suma_Sprzedazy", ascending=False)
raport_kraj

---
### FINAŁOWY CASE STUDY: Raport Rentowności E-Commerce (20 min)
Jesteś analitykiem danych w międzynarodowym sklepie internetowym. Zarząd poprosił Cię o raport dotyczący zyskowności poszczególnych rynków i kategorii.

Oto wygenerowana baza transakcji z ostatniego miesiąca:

In [ ]:
# Generowanie zbioru danych biznesowych (uruchom tę komórkę)
np.random.seed(42)
n_rows = 100

kategorie = ["Laptopy", "Smartfony", "Akcesoria", "Monitory"]
kraje = ["Polska", "Niemcy", "Czechy", "Francja"]

dane_case_study = {
    "Transakcja_ID": np.arange(1001, 1001 + n_rows),
    "Kraj": np.random.choice(kraje, n_rows, p=[0.4, 0.3, 0.15, 0.15]),
    "Kategoria": np.random.choice(kategorie, n_rows),
    "Przychód_PLN": np.random.uniform(50, 5000, n_rows).round(2),
    "Koszt_Wytworzenia_PLN": np.random.uniform(30, 3500, n_rows).round(2),
    "Ocena_Klienta_1_5": np.random.choice([1, 2, 3, 4, 5, np.nan], n_rows, p=[0.05, 0.05, 0.2, 0.3, 0.35, 0.05])
}

# Celowo wstrzykujemy kilka zduplikowanych transakcji
df_ecommerce = pd.DataFrame(dane_case_study)
df_ecommerce = pd.concat([df_ecommerce, df_ecommerce.iloc[0:3]], ignore_index=True)

print(f"Wygenerowano zbiór e-commerce o wymiarach: {df_ecommerce.shape}")
df_ecommerce.head()

#### Polecenia do wykonania w ramach Case Study:
1. **Czyszczenie danych:**
   - Usuń duplikaty z tabeli.
   - Sprawdź braki w `Ocena_Klienta_1_5` i uzupełnij je zaokrągloną średnią oceną ze wszystkich transakcji.
2. **Feature Engineering:**
   - Oblicz kolumnę `Zysk_PLN` = `Przychód_PLN - Koszt_Wytworzenia_PLN`.
   - Oblicz kolumnę `Marza_Procent` = `(Zysk_PLN / Przychód_PLN) * 100`.
3. **Raport Biznesowy dla Zarządu:**
   - Zgrupuj dane według **Kategorii** i oblicz: sumę przychodów, łączny zysk oraz średnią ocenę klienta.
   - Posortuj kategorie od generującej **największy łączny zysk**.
   - Wskaż, który **Kraj** osiąga najwyższą średnią marżę procentową.

In [ ]:
# [ROZWIĄZANIE CASE STUDY]

# KROK 1: Czyszczenie
df_case = df_ecommerce.drop_duplicates().copy()
srednia_ocena = round(df_case["Ocena_Klienta_1_5"].mean(), 1)
df_case["Ocena_Klienta_1_5"] = df_case["Ocena_Klienta_1_5"].fillna(srednia_ocena)

# KROK 2: Nowe metryki
df_case["Zysk_PLN"] = df_case["Przychód_PLN"] - df_case["Koszt_Wytworzenia_PLN"]
df_case["Marza_Procent"] = (df_case["Zysk_PLN"] / df_case["Przychód_PLN"]) * 100

# KROK 3: Raport wg Kategorii
raport_kategorie = df_case.groupby("Kategoria").agg(
    Suma_Przychodow=("Przychód_PLN", "sum"),
    Laczny_Zysk=("Zysk_PLN", "sum"),
    Srednia_Ocena=("Ocena_Klienta_1_5", "mean")
).sort_values(by="Laczny_Zysk", ascending=False).round(2)

print("=== RAPORT ZYSKOWNOŚCI WG KATEGORII ===")
print(raport_kategorie)

# KROK 4: Marża wg Krajów
raport_kraje = df_case.groupby("Kraj")["Marza_Procent"].mean().sort_values(ascending=False).round(2)
print("\n=== ŚREDNIA MARŻA PROCENTOWA WG KRAJÓW ===")
print(raport_kraje)

### Podsumowanie Laboratorium 2
Opanowaliśmy fundamenty inżynierii danych:
- Wektorowe obliczenia w **NumPy**.
- Selekcję, filtrowanie i indeksowanie w **Pandas**.
- Wykrywanie braków i czyszczenie "brudnych" danych.
- Agregacje grupowe (`groupby`), które są podstawą analityki biznesowej (BI).

Na **Laboratorium 3** połączymy te umiejętności z **wizualizacją (Matplotlib / Seaborn)**, zrobimy pełne **EDA (Exploratory Data Analysis)** i wytrenujemy **nasz pierwszy model Uczenia Maszynowego (Scikit-learn)**!

---
# ZADANIE PUNKTOWANE: Laboratorium 2 (Ocena Cząstkowa – max 10 pkt)
**Czas na wykonanie:** 20 minut na zajęciach (lub do końca dnia)  
**Scenariusz biznesowy:** Bankowy Dział Kart Kredytowych  

Otrzymujesz surowy zbiór danych dotyczący aktywności klientów korzystających z kart kredytowych. 
Twoim zadaniem jest uporządkowanie danych, policzenie wskaźników ryzyka i przygotowanie zestawienia dla Dyrektora Ryzyka.

In [ ]:
# [URUCHOM TĘ KOMÓRKĘ ABY WYGENEROWAĆ SWÓJ ZBIÓR DANYCH DO ZADANIA]
import numpy as np
import pandas as pd

np.random.seed(123)
n = 80

dane_bankowe = {
    "Klient_ID": [f"K_{i}" for i in range(1001, 1001 + n)],
    "Segment": np.random.choice(["Standard", "Silver", "Gold", "VIP"], n, p=[0.4, 0.3, 0.2, 0.1]),
    "Limit_Karty_PLN": np.random.choice([3000, 5000, 10000, 20000, np.nan], n, p=[0.3, 0.3, 0.2, 0.15, 0.05]),
    "Aktualne_Zadluzenie_PLN": np.random.uniform(500, 18000, n).round(2),
    "Liczba_Transakcji_Msc": np.random.choice([5, 12, 25, 40, np.nan], n, p=[0.2, 0.4, 0.25, 0.1, 0.05]),
    "Czy_Zalega": np.random.choice([0, 1], n, p=[0.85, 0.15])
}

df_bank = pd.DataFrame(dane_bankowe)
# Wstrzykujemy 3 duplikaty
df_bank = pd.concat([df_bank, df_bank.iloc[0:3]], ignore_index=True)

print("Zbiór wygenerowany! Rozpocznij pracę poniżej.")
df_bank.head()

### Treść poleceń i punktacja:

1. **Czyszczenie danych (2 pkt):**
   - Usuń duplikaty z tabeli.
   - Uzupełnij braki w `Limit_Karty_PLN` medianą tej kolumny.
   - Usuń wiersze, w których brakuje `Liczba_Transakcji_Msc`.

2. **Feature Engineering – Wskaźnik Ryzyka (3 pkt):**
   - Utwórz kolumnę `Wykorzystanie_Limitu_Proc` = `(Aktualne_Zadluzenie_PLN / Limit_Karty_PLN) * 100`. Zaokrąglij do 1 miejsca po przecinku.
   - Utwórz kolumnę logiczną (True/False) `Wysokie_Ryzyko`:
     Klient ma wysokie ryzyko, jeśli `Wykorzystanie_Limitu_Proc > 80%` **LUB** `Czy_Zalega == 1`.

3. **Selekcja i Filtrowanie (2 pkt):**
   - Z ilu klientów o statusie `Wysokie_Ryzyko == True` składa się baza? (Wypisz przefiltrowaną tabelę lub użyj `.shape[0]`).

4. **Raport Biznesowy (3 pkt):**
   - Pogrupuj dane według **Segmentu** klienta i oblicz:
     - Średni limit karty (`Limit_Karty_PLN`).
     - Średnie wykorzystanie limitu (`Wykorzystanie_Limitu_Proc`).
     - Liczbę klientów w danym segmencie.
   - Posortuj wynik malejąco według średniego wykorzystania limitu.

In [ ]:
# ==========================================
# ROZWIĄZANIE DLA PROWADZĄCEGO
# ==========================================

# Krok 1 (2 pkt)
df_clean = df_bank.drop_duplicates().copy()
print(f"Liczba wierszy: {len(df_clean)}")

df_clean["Limit_Karty_PLN"] = df_clean["Limit_Karty_PLN"].fillna(df_clean["Limit_Karty_PLN"].median())
df_clean = df_clean.dropna(subset=["Liczba_Transakcji_Msc"])
print(f"Liczba wierszy po czyszczeniu: {len(df_clean)}")


# Krok 2 (3 pkt)
df_clean["Wykorzystanie_Limitu_Proc"] = np.round((df_clean["Aktualne_Zadluzenie_PLN"] / df_clean["Limit_Karty_PLN"]) * 100, 1)
df_clean["Wysokie_Ryzyko"] = (df_clean["Wykorzystanie_Limitu_Proc"] > 80.0) | (df_clean["Czy_Zalega"] == 1)

# Krok 3 (2 pkt)
ryzykowni_klienci = df_clean[df_clean["Wysokie_Ryzyko"] == True]
liczba_ryzykownych = len(ryzykowni_klienci)
print(f"Liczba ryzykownych klientów: {liczba_ryzykownych}")

# Krok 4 (3 pkt)
raport_segmenty = df_clean.groupby("Segment").agg(
    Sredni_Limit=('Limit_Karty_PLN', 'mean'),
    Srednie_Wykorzystanie_Proc=('Wykorzystanie_Limitu_Proc', 'mean'),
    Liczba_Klientow=('Klient_ID', 'count')
).sort_values(by="Srednie_Wykorzystanie_Proc", ascending=False).round(2)

print("\nRaport końcowy:")
print(raport_segmenty)